<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_3_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.3-A
# EXACT vs MONTE CARLO TEACHING AT FIXED R
#
# RESUMABLE / GOOGLE-DRIVE / CPU-ONLY VERSION
#
# R = 2000
# m = 1, 5, 10, 25
#
# Nested Monte Carlo:
#     actual nominal trajectories = 2000 * 25 = 50,000
#
# IMPORTANT:
#   - same 25 paths/configuration generate m=1,5,10,25 labels
#   - MC stops immediately when C>N because overflow bin is already known
#
# PERSISTENCE:
#   - exact targets saved in permanent Drive chunks
#   - MC simulations saved in permanent Drive chunks
#   - neural checkpoint every 5 epochs
#   - optimizer + scheduler + RNG state saved
#   - completed models permanently saved
#   - completed results permanently saved
#
# AFTER COLAB DISCONNECT:
#   simply run THIS SAME CELL again
#
# SCIENTIFIC OUTPUTS:
#   1 final table
#   1 two-panel figure
#
# CPU ONLY
# SPARSE LU ONLY
# NO MATRIX INVERSE
# =====================================================================================


# =====================================================================================
# 0. MOUNT GOOGLE DRIVE
# =====================================================================================

from google.colab import drive
drive.mount(
    "/content/drive",
    force_remount=False
)


# =====================================================================================
# 1. CPU ENVIRONMENT
# =====================================================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


# =====================================================================================
# 2. IMPORTS
# =====================================================================================

import time
import math
import random
import pickle
import shutil

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

from joblib import Parallel, delayed

from numba import (
    njit,
    prange,
    set_num_threads,
    get_num_threads
)

import torch
import torch.nn as nn

import matplotlib.pyplot as plt


# =====================================================================================
# 3. CONFIGURATION
# =====================================================================================

@dataclass
class C:

    seed:int = 20260820

    beta:Tuple[float,float] = (.30,1.50)
    gamma:Tuple[float,float] = (.20,1.00)
    omega:Tuple[float,float] = (.02,.50)
    frac:Tuple[float,float] = (.02,.20)

    trainN:Tuple[int,...] = tuple(
        range(40,401,20)
    )

    R:int = 2000

    nval:int = 400
    ntest:int = 700

    width:int = 128
    depth:int = 3

    batch:int = 64
    epochs:int = 500

    lr:float = 1e-3
    wd:float = 1e-6

    patience:int = 20
    min_delta:float = 1e-6
    clip:float = 5.


cfg = C()


# =====================================================================================
# 4. MONTE CARLO DESIGN
# =====================================================================================

M_VALUES = (
    1,
    5,
    10,
    25
)

TOTAL_MC_EPISODES = (
    cfg.R
    *
    max(M_VALUES)
)

assert (
    TOTAL_MC_EPISODES
    ==
    50_000
)

Nscale = max(
    cfg.trainN
)

TEST_N = tuple(
    range(40,401,10)
)


# =====================================================================================
# 5. PERMANENT GOOGLE DRIVE STORAGE
# =====================================================================================

ROOT = Path(
    "/content/drive/MyDrive/"
    "StatisticalLearning/"
    "Experiment_5_3A_R2000_MC50000_v4"
)

CACHE = ROOT / "cache"

EXACT_CACHE = CACHE / "exact"
MC_CACHE = CACHE / "mc"
MODEL_CACHE = CACHE / "models"
CHECKPOINT_CACHE = CACHE / "checkpoints"

OUT = ROOT / "results"


for d in (
    ROOT,
    CACHE,
    EXACT_CACHE,
    MC_CACHE,
    MODEL_CACHE,
    CHECKPOINT_CACHE,
    OUT
):

    d.mkdir(
        parents=True,
        exist_ok=True
    )


print("="*95)
print("PERMANENT GOOGLE DRIVE DIRECTORY")
print(ROOT)
print("="*95)


# =====================================================================================
# 6. REUSE EXACT RESULTS FROM PREVIOUS 10,000-MC EXPERIMENT, IF AVAILABLE
#
# Exact train/validation/test data are identical.
# The exact-teacher neural model is also identical.
# We do NOT copy previous MC labels/models.
# =====================================================================================

PREVIOUS_ROOT = Path(
    "/content/drive/MyDrive/"
    "StatisticalLearning/"
    "Experiment_5_3A_R2000_MC10000_v3"
)

PREVIOUS_EXACT = (
    PREVIOUS_ROOT
    /
    "cache"
    /
    "exact"
)

PREVIOUS_MODELS = (
    PREVIOUS_ROOT
    /
    "cache"
    /
    "models"
)


if PREVIOUS_ROOT.exists():

    print(
        "\nPrevious 5.3-A Drive experiment found."
    )

    print(
        "Reusing compatible exact computations where possible..."
    )


    for name in (
        "TRAIN_full.pkl",
        "VALIDATION_full.pkl",
        "TEST_full.pkl"
    ):

        old = PREVIOUS_EXACT / name
        new = EXACT_CACHE / name

        if (
            old.exists()
            and
            not new.exists()
        ):

            shutil.copy2(
                old,
                new
            )

            print(
                "Copied:",
                name
            )


    old_model = (
        PREVIOUS_MODELS
        /
        "EXACT_R2000_FINAL.pt"
    )

    new_model = (
        MODEL_CACHE
        /
        "EXACT_R2000_FINAL.pt"
    )

    if (
        old_model.exists()
        and
        not new_model.exists()
    ):

        shutil.copy2(
            old_model,
            new_model
        )

        print(
            "Copied exact-teacher model."
        )


# =====================================================================================
# 7. ATOMIC SAVING
#
# Prevent corrupted files if Colab disconnects while writing.
# =====================================================================================

def atomic_pickle(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        tmp,
        "wb"
    ) as f:

        pickle.dump(
            obj,
            f,
            pickle.HIGHEST_PROTOCOL
        )

    os.replace(
        tmp,
        path
    )


def atomic_torch_save(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        obj,
        tmp
    )

    os.replace(
        tmp,
        path
    )


def safe_pickle_load(
    path,
    default=None
):

    try:

        with open(
            path,
            "rb"
        ) as f:

            return pickle.load(f)

    except Exception:

        return default


# =====================================================================================
# 8. CPU HARDWARE
# =====================================================================================

CPU = (
    os.cpu_count()
    or 1
)


# Sparse LU is RAM-intensive
N_EXACT = max(
    1,
    min(
        2,
        CPU
    )
)


# Numba Monte Carlo
N_MC = max(
    1,
    min(
        CPU,
        get_num_threads()
    )
)

set_num_threads(
    N_MC
)


# PyTorch CPU
TORCH_THREADS = max(
    1,
    min(
        8,
        CPU
    )
)

torch.set_num_threads(
    TORCH_THREADS
)


try:

    torch.set_num_interop_threads(
        1
    )

except RuntimeError:

    pass


def seed_all(s):

    random.seed(s)

    np.random.seed(s)

    torch.manual_seed(s)


seed_all(
    cfg.seed
)


print(
    "CPU cores:",
    CPU
)

print(
    "Exact sparse-LU workers:",
    N_EXACT
)

print(
    "Numba threads:",
    N_MC
)

print(
    "PyTorch threads:",
    torch.get_num_threads()
)

print(
    "Training configurations R:",
    cfg.R
)

print(
    "Monte Carlo m values:",
    M_VALUES
)

print(
    "Nested MC trajectories:",
    f"{TOTAL_MC_EPISODES:,}"
)


# =====================================================================================
# 9. RECORD
# =====================================================================================

@dataclass
class Rec:

    b:float
    g:float
    w:float

    N:int
    i0:int

    p:np.ndarray


# =====================================================================================
# 10. SIRS STATE-SPACE TOPOLOGY
# =====================================================================================

@lru_cache(None)
def topo(N):

    states = [

        (s,i)

        for i in range(
            1,
            N+1
        )

        for s in range(
            N-i+1
        )

    ]


    ix = {

        x:j

        for j,x
        in enumerate(states)

    }


    M = len(states)


    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]


    db = np.zeros(
        M,
        dtype=np.float64
    )

    dg = np.zeros(
        M,
        dtype=np.float64
    )

    dw = np.zeros(
        M,
        dtype=np.float64
    )

    qb = np.zeros(
        M,
        dtype=np.float64
    )


    for j,(s,i) in enumerate(states):

        r = (
            N-s-i
        )


        # ------------------------------------------------
        # Infection
        # ------------------------------------------------

        if s:

            ir.append(j)

            ic.append(
                ix[
                    (s-1,i+1)
                ]
            )

            rate = (
                s*i/N
            )

            ib.append(
                rate
            )

            db[j] = rate


        # ------------------------------------------------
        # Recovery
        # ------------------------------------------------

        dg[j] = i


        if i == 1:

            qb[j] = i

        else:

            rr.append(j)

            rc.append(
                ix[
                    (s,i-1)
                ]
            )

            rb.append(i)


        # ------------------------------------------------
        # Immunity loss
        # ------------------------------------------------

        if r:

            wr.append(j)

            wc.append(
                ix[
                    (s+1,i)
                ]
            )

            wb.append(r)

            dw[j] = r


    A = lambda x,d=float: np.asarray(
        x,
        dtype=d
    )


    return (

        ix,
        M,

        A(ir,int),
        A(ic,int),
        A(ib),

        A(rr,int),
        A(rc,int),
        A(rb),

        A(wr,int),
        A(wc,int),
        A(wb),

        db,
        dg,
        dw,
        qb
    )


# =====================================================================================
# 11. EXACT INFECTION-COUNT DISTRIBUTION
# =====================================================================================

def exact_p(
    b,
    g,
    w,
    N,
    i0
):

    (
        ix,
        M,

        ir,
        ic,
        ib,

        rr,
        rc,
        rb,

        wr,
        wc,
        wb,

        db,
        dg,
        dw,
        qb

    ) = topo(N)


    rows = np.r_[

        ir,
        rr,
        wr,
        np.arange(M)

    ]


    cols = np.r_[

        ic,
        rc,
        wc,
        np.arange(M)

    ]


    vals = np.r_[

        b*ib,

        g*rb,

        w*wb,

        -(
            b*db
            +
            g*dg
            +
            w*dw
        )

    ]


    T = sparse.coo_matrix(

        (
            vals,
            (
                rows,
                cols
            )
        ),

        shape=(
            M,
            M
        ),

        dtype=np.float64

    ).tocsc()


    D1 = sparse.coo_matrix(

        (
            b*ib,
            (
                ir,
                ic
            )
        ),

        shape=(
            M,
            M
        ),

        dtype=np.float64

    ).tocsc()


    D0 = (
        T-D1
    ).tocsc()


    A0 = (
        -D0
    ).tocsc()


    lu = splu(
        A0,
        permc_spec="COLAMD"
    )


    q = (
        g*qb
    )


    initial = ix[
        (
            N-i0,
            i0
        )
    ]


    v = np.zeros(
        M,
        dtype=np.float64
    )

    v[
        initial
    ] = 1.


    bvec = lu.solve(
        q
    )


    D1T = D1.T.tocsr()


    p = np.zeros(
        N+2,
        dtype=np.float64
    )


    for k in range(
        N+1
    ):

        p[k] = (
            v @ bvec
        )


        v = np.asarray(

            D1T

            @

            lu.solve(
                v,
                trans="T"
            )

        ).ravel()


    # overflow
    p[-1] = (
        v.sum()
    )


    p[
        np.abs(p)<1e-10
    ] = 0.


    p = np.maximum(
        p,
        0.
    )


    mass = (
        p.sum()
    )


    if (
        not np.isfinite(mass)
        or
        mass<=0
    ):

        raise RuntimeError(
            f"Invalid exact PMF: "
            f"N={N}, i0={i0}"
        )


    return (
        p/mass
    )


# =====================================================================================
# 12. EXPERIMENTAL DESIGN
# =====================================================================================

def design(
    n,
    Ns,
    seed
):

    U = qmc.LatinHypercube(

        d=4,

        seed=seed

    ).random(
        n
    )


    scale = lambda x,a: (

        a[0]

        +

        (
            a[1]-a[0]
        )
        *
        x

    )


    beta = scale(
        U[:,0],
        cfg.beta
    )


    gamma = scale(
        U[:,1],
        cfg.gamma
    )


    omega = scale(
        U[:,2],
        cfg.omega
    )


    frac = scale(
        U[:,3],
        cfg.frac
    )


    Nv = np.tile(

        np.asarray(
            Ns
        ),

        math.ceil(
            n/len(Ns)
        )

    )[:n]


    rng = np.random.default_rng(
        seed+991
    )


    rng.shuffle(
        Nv
    )


    i0 = np.asarray([

        int(
            np.clip(
                round(
                    frac[j]
                    *
                    Nv[j]
                ),
                2,
                Nv[j]
            )
        )

        for j in range(n)

    ])


    for N in Ns:

        z = np.where(
            Nv==N
        )[0]


        if len(z)==0:

            continue


        k = max(
            1,
            round(
                .25*len(z)
            )
        )


        i0[
            rng.choice(
                z,
                k,
                replace=False
            )
        ] = 1


    return [

        (
            float(beta[j]),
            float(gamma[j]),
            float(omega[j]),
            int(Nv[j]),
            int(i0[j])
        )

        for j in range(n)

    ]


# =====================================================================================
# 13. RESUMABLE EXACT TARGET GENERATION
#
# Smaller chunks reduce work lost after a disconnect.
# =====================================================================================

EXACT_CHUNK = 50


def _exact_one(
    j,
    x
):

    return (

        j,

        Rec(
            *x,
            exact_p(
                *x
            )
        )

    )


def exact_set_resumable(
    configs,
    name
):

    full_file = (
        EXACT_CACHE
        /
        f"{name}_full.pkl"
    )


    # -------------------------------------------------------------------------
    # Completed dataset
    # -------------------------------------------------------------------------

    if full_file.exists():

        ans = safe_pickle_load(
            full_file
        )


        if (
            ans is not None
            and
            len(ans)==len(configs)
        ):

            print(
                f"{name}: full Drive cache loaded "
                f"({len(ans):,})"
            )

            return ans


    folder = (
        EXACT_CACHE
        /
        name
    )


    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    ans = []


    for start in range(
        0,
        len(configs),
        EXACT_CHUNK
    ):

        end = min(
            start+EXACT_CHUNK,
            len(configs)
        )


        file = (
            folder
            /
            f"chunk_{start:05d}_{end:05d}.pkl"
        )


        part = None


        # ---------------------------------------------------------------------
        # Existing chunk
        # ---------------------------------------------------------------------

        if file.exists():

            part = safe_pickle_load(
                file
            )


            if (
                part is not None
                and
                len(part)==end-start
            ):

                print(
                    f"{name} "
                    f"{start:5d}:{end:5d} | "
                    "Drive cache"
                )

            else:

                part = None


        # ---------------------------------------------------------------------
        # Compute missing chunk
        # ---------------------------------------------------------------------

        if part is None:

            print(
                f"{name} "
                f"{start:5d}:{end:5d} | "
                f"computing with {N_EXACT} workers"
            )


            jobs = list(
                enumerate(
                    configs[
                        start:end
                    ]
                )
            )


            jobs.sort(
                key=lambda x:
                x[1][3],
                reverse=True
            )


            t0 = time.perf_counter()


            z = Parallel(

                n_jobs=N_EXACT,

                backend="threading"

            )(

                delayed(
                    _exact_one
                )(
                    j,
                    x
                )

                for j,x in jobs

            )


            z.sort(
                key=lambda x:
                x[0]
            )


            part = [
                r
                for _,r in z
            ]


            atomic_pickle(
                part,
                file
            )


            print(
                f"saved permanently | "
                f"{time.perf_counter()-t0:.1f}s"
            )


        ans.extend(
            part
        )


    # -------------------------------------------------------------------------
    # Save complete assembled dataset
    # -------------------------------------------------------------------------

    atomic_pickle(
        ans,
        full_file
    )


    print(
        f"{name}: COMPLETE and permanently cached."
    )


    return ans


# =====================================================================================
# 14. NUMBA SIRS MONTE CARLO
#
# EXACT early stopping:
#
# We need the truncated-with-overflow distribution
#
#   p(0),...,p(N),p(>N).
#
# Once C=N+1, the trajectory is definitely in the overflow bin.
# Continuing to extinction cannot change its label.
# =====================================================================================

@njit
def sim_C_numba(
    b,
    g,
    w,
    N,
    i0
):

    S = (
        N-i0
    )

    I = i0

    R = 0

    C = 0


    while I > 0:

        infection = (
            b*S*I/N
        )

        recovery = (
            g*I
        )

        waning = (
            w*R
        )


        total = (
            infection
            +
            recovery
            +
            waning
        )


        z = (
            np.random.random()
            *
            total
        )


        if z < infection:

            S -= 1
            I += 1
            C += 1


            # ---------------------------------------------------------
            # Exact overflow termination
            # ---------------------------------------------------------

            if C >= N+1:

                return N+1


        elif z < (
            infection
            +
            recovery
        ):

            I -= 1
            R += 1


        else:

            R -= 1
            S += 1


    return C


# =====================================================================================
# 15. NESTED MC CHUNK
#
# m = 1,5,10,25
# =====================================================================================

MC_LEVELS = np.asarray(
    [
        1,
        5,
        10,
        25
    ],
    dtype=np.int64
)


@njit(parallel=True)
def mc_nested_chunk(
    B,
    G,
    W,
    N,
    I0,
    maxN,
    global_start,
    seed
):

    n = len(N)


    levels = np.asarray(
        [
            1,
            5,
            10,
            25
        ],
        dtype=np.int64
    )


    out = np.zeros(

        (
            4,
            n,
            maxN+2
        ),

        dtype=np.int16

    )


    for j in prange(n):

        global_j = (
            global_start+j
        )


        np.random.seed(
            seed
            +
            100003*global_j
        )


        hist = np.zeros(
            maxN+2,
            dtype=np.int16
        )


        level = 0


        # -------------------------------------------------------------
        # Exactly 25 nested simulations/configuration
        # -------------------------------------------------------------

        for m in range(
            1,
            26
        ):

            c = sim_C_numba(

                B[j],
                G[j],
                W[j],
                N[j],
                I0[j]

            )


            hist[c] += 1


            if (
                level < 4
                and
                m == levels[level]
            ):

                out[
                    level,
                    j,
                    :
                ] = hist


                level += 1


    return out


# =====================================================================================
# 16. RESUMABLE MONTE CARLO
#
# Saved in chunks permanently to Google Drive.
# =====================================================================================

MC_CHUNK = 100


def mc_labels_resumable(
    records
):

    final_file = (

        MC_CACHE

        /

        "MC_nested_R2000_m25_50000_overflow_v4.pkl"

    )


    # -------------------------------------------------------------------------
    # Complete MC labels already exist
    # -------------------------------------------------------------------------

    if final_file.exists():

        ans = safe_pickle_load(
            final_file
        )


        if ans is not None:

            print(
                "Final 50,000-trajectory MC labels "
                "loaded from Drive."
            )

            return ans


    # -------------------------------------------------------------------------
    # Arrays
    # -------------------------------------------------------------------------

    B = np.asarray(
        [
            r.b
            for r in records
        ],
        dtype=np.float64
    )


    G = np.asarray(
        [
            r.g
            for r in records
        ],
        dtype=np.float64
    )


    W = np.asarray(
        [
            r.w
            for r in records
        ],
        dtype=np.float64
    )


    N = np.asarray(
        [
            r.N
            for r in records
        ],
        dtype=np.int64
    )


    I0 = np.asarray(
        [
            r.i0
            for r in records
        ],
        dtype=np.int64
    )


    print("\n"+"="*90)
    print("RESUMABLE NESTED MONTE CARLO")
    print("="*90)

    print(
        "R:",
        cfg.R
    )

    print(
        "m values:",
        M_VALUES
    )

    print(
        "Maximum simulations/configuration:",
        max(M_VALUES)
    )

    print(
        "Nominal actual MC trajectories:",
        f"{TOTAL_MC_EPISODES:,}"
    )

    print(
        "Early overflow termination:",
        "ON"
    )

    print(
        "MC chunk size:",
        MC_CHUNK
    )


    # -------------------------------------------------------------------------
    # Numba compilation
    # -------------------------------------------------------------------------

    print(
        "\nCompiling Numba MC kernel..."
    )


    _ = mc_nested_chunk(

        np.asarray(
            [.8],
            dtype=np.float64
        ),

        np.asarray(
            [.5],
            dtype=np.float64
        ),

        np.asarray(
            [.1],
            dtype=np.float64
        ),

        np.asarray(
            [40],
            dtype=np.int64
        ),

        np.asarray(
            [1],
            dtype=np.int64
        ),

        40,

        0,

        cfg.seed+20

    )


    print(
        "Numba compilation complete."
    )


    pieces = []


    # -------------------------------------------------------------------------
    # Chunked simulation
    # -------------------------------------------------------------------------

    for start in range(
        0,
        cfg.R,
        MC_CHUNK
    ):

        end = min(
            start+MC_CHUNK,
            cfg.R
        )


        file = (

            MC_CACHE

            /

            f"chunk_{start:05d}_{end:05d}_"
            f"m25_overflow_v4.pkl"

        )


        part = None


        if file.exists():

            part = safe_pickle_load(
                file
            )


            if (
                part is not None
                and
                part.shape[1]==end-start
            ):

                print(
                    f"MC "
                    f"{start:5d}:{end:5d} | "
                    "Drive cache"
                )

            else:

                part = None


        if part is None:

            t0 = time.perf_counter()


            part = mc_nested_chunk(

                B[start:end],
                G[start:end],
                W[start:end],
                N[start:end],
                I0[start:end],

                max(
                    cfg.trainN
                ),

                start,

                cfg.seed+20

            )


            atomic_pickle(
                part,
                file
            )


            print(
                f"MC "
                f"{start:5d}:{end:5d} | "
                f"{time.perf_counter()-t0:.1f}s | "
                "saved permanently"
            )


        pieces.append(
            part
        )


    # -------------------------------------------------------------------------
    # Assemble all chunks
    # -------------------------------------------------------------------------

    raw = np.concatenate(
        pieces,
        axis=1
    )


    level = {
        1:0,
        5:1,
        10:2,
        25:3
    }


    labels = {}


    for m in M_VALUES:

        q = level[m]


        labels[m] = [

            raw[
                q,
                j,
                :records[j].N+2
            ].astype(
                np.float32
            )
            /
            m

            for j in range(
                len(records)
            )

        ]


    atomic_pickle(
        labels,
        final_file
    )


    print(
        "\n50,000-trajectory MC labels "
        "COMPLETE and permanently cached."
    )


    del raw
    del pieces


    return labels


# =====================================================================================
# 17. HAZARD NETWORK
# =====================================================================================

class HazardNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        layers = []

        d = 6


        for _ in range(
            cfg.depth
        ):

            layers += [

                nn.Linear(
                    d,
                    cfg.width
                ),

                nn.SiLU()

            ]


            d = (
                cfg.width
            )


        layers.append(

            nn.Linear(
                d,
                1
            )

        )


        self.net = nn.Sequential(
            *layers
        )


    def forward(
        self,
        x
    ):

        return torch.sigmoid(

            self.net(
                x
            ).squeeze(-1)

        )


# =====================================================================================
# 18. PACK SAME-N DATA ON CPU
# =====================================================================================

def pack(
    records,
    labels=None
):

    groups = {}


    for j,r in enumerate(records):

        groups.setdefault(
            r.N,
            []
        ).append(j)


    P = {}


    for N,idx in groups.items():

        idx = np.asarray(
            idx,
            dtype=int
        )


        B = len(idx)

        K = N+1


        beta = torch.tensor(

            [
                records[j].b
                for j in idx
            ],

            dtype=torch.float32

        )[:,None]


        gamma = torch.tensor(

            [
                records[j].g
                for j in idx
            ],

            dtype=torch.float32

        )[:,None]


        omega = torch.tensor(

            [
                records[j].w
                for j in idx
            ],

            dtype=torch.float32

        )[:,None]


        ns = torch.full(

            (
                B,
                1
            ),

            N/Nscale,

            dtype=torch.float32

        )


        i0 = torch.tensor(

            [
                records[j].i0/N
                for j in idx
            ],

            dtype=torch.float32

        )[:,None]


        c = (

            torch.arange(
                K,
                dtype=torch.float32
            )

            /

            N

        )[None,:]


        X = torch.stack(

            [

                beta.expand(B,K),

                gamma.expand(B,K),

                omega.expand(B,K),

                ns.expand(B,K),

                i0.expand(B,K),

                c.expand(B,K)

            ],

            dim=2

        ).contiguous()


        Y_np = np.stack([

            records[j].p

            if labels is None

            else labels[j]

            for j in idx

        ]).astype(
            np.float32,
            copy=False
        )


        P[N] = {

            "X":
                X,

            "Y":
                torch.from_numpy(
                    Y_np
                ),

            "n":
                B

        }


    return P


# =====================================================================================
# 19. HAZARD -> PMF
# =====================================================================================

def reconstruct_batch(
    h
):

    B = (
        h.shape[0]
    )


    before = torch.cat(

        [

            torch.ones(
                (B,1),
                dtype=h.dtype
            ),

            torch.cumprod(
                1-h[:,:-1],
                dim=1
            )

        ],

        dim=1

    )


    pmf = (
        before*h
    )


    overflow = torch.prod(

        1-h,

        dim=1,

        keepdim=True

    )


    return torch.cat(

        [
            pmf,
            overflow
        ],

        dim=1

    )


# =====================================================================================
# 20. TAIL-RISK
# =====================================================================================

def tail_batch(
    P
):

    return torch.flip(

        torch.cumsum(

            torch.flip(
                P[:,1:],
                dims=[1]
            ),

            dim=1

        ),

        dims=[1]

    )


# =====================================================================================
# 21. CPU-VECTORIZED FORWARD
# =====================================================================================

def predict_batch(
    net,
    X
):

    B,K,_ = (
        X.shape
    )


    h = net(

        X.reshape(
            B*K,
            6
        )

    ).reshape(
        B,
        K
    )


    return reconstruct_batch(
        h
    )


# =====================================================================================
# 22. LOSS
# =====================================================================================

def batch_loss(
    net,
    X,
    Y
):

    P = predict_batch(
        net,
        X
    )


    Lp = torch.sum(

        (
            P-Y
        )**2,

        dim=1

    )


    Lrho = torch.mean(

        (

            tail_batch(P)

            -

            tail_batch(Y)

        )**2,

        dim=1

    )


    return (
        Lp
        +
        Lrho
    ).mean()


# =====================================================================================
# 23. SAME-N MINI-BATCH SCHEDULE
# =====================================================================================

def schedule(
    P,
    rng,
    shuffle=True
):

    ans = []


    for N,G in P.items():

        idx = np.arange(
            G["n"]
        )


        if shuffle:

            rng.shuffle(
                idx
            )


        for s in range(
            0,
            len(idx),
            cfg.batch
        ):

            ans.append(

                (

                    N,

                    idx[
                        s:
                        s+cfg.batch
                    ]

                )

            )


    if shuffle:

        rng.shuffle(
            ans
        )


    return ans


# =====================================================================================
# 24. VALIDATION
# =====================================================================================

@torch.no_grad()
def val_loss(
    net,
    P
):

    net.eval()


    total = 0.

    n = 0


    for N,G in P.items():

        for s in range(
            0,
            G["n"],
            cfg.batch
        ):

            X = G["X"][
                s:
                s+cfg.batch
            ]


            Y = G["Y"][
                s:
                s+cfg.batch
            ]


            L = batch_loss(
                net,
                X,
                Y
            )


            B = len(X)


            total += (
                L.item()
                *
                B
            )


            n += B


    return (
        total/n
    )


# =====================================================================================
# 25. RESUMABLE CPU TRAINING
#
# Permanent checkpoint every 5 epochs.
# =====================================================================================

CHECKPOINT_EVERY = 5


def fit_resumable(
    TR,
    VA,
    name,
    seed
):

    final_file = (

        MODEL_CACHE

        /

        f"{name}_FINAL.pt"

    )


    ckpt_file = (

        CHECKPOINT_CACHE

        /

        f"{name}_CHECKPOINT.pt"

    )


    # -------------------------------------------------------------------------
    # Finished model
    # -------------------------------------------------------------------------

    if final_file.exists():

        z = torch.load(

            final_file,

            map_location="cpu",

            weights_only=False

        )


        net = HazardNet()


        net.load_state_dict(
            z["state"]
        )


        best_epoch = z.get(
            "best_epoch",
            z.get(
                "epoch",
                -1
            )
        )


        print(
            f"{name}: FINAL model loaded | "
            f"best epoch={best_epoch}"
        )


        return (
            net,
            z.get(
                "training_sec",
                0.
            )
        )


    # -------------------------------------------------------------------------
    # Initial state
    # -------------------------------------------------------------------------

    seed_all(
        seed
    )


    net = HazardNet()


    opt = torch.optim.AdamW(

        net.parameters(),

        lr=cfg.lr,

        weight_decay=cfg.wd

    )


    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(

            opt,

            factor=.5,

            patience=15

        )
    )


    rng = np.random.default_rng(
        seed+8
    )


    start_epoch = 1

    best = np.inf

    best_epoch = 0

    best_state = None

    wait = 0

    previous_elapsed = 0.


    # -------------------------------------------------------------------------
    # Resume checkpoint
    # -------------------------------------------------------------------------

    if ckpt_file.exists():

        try:

            ck = torch.load(

                ckpt_file,

                map_location="cpu",

                weights_only=False

            )


            net.load_state_dict(
                ck["model"]
            )


            opt.load_state_dict(
                ck["optimizer"]
            )


            scheduler.load_state_dict(
                ck["scheduler"]
            )


            start_epoch = (
                ck["epoch"]
                +
                1
            )


            best = (
                ck["best"]
            )


            best_epoch = (
                ck["best_epoch"]
            )


            best_state = (
                ck["best_state"]
            )


            wait = (
                ck["wait"]
            )


            previous_elapsed = (
                ck.get(
                    "elapsed_sec",
                    0.
                )
            )


            rng.bit_generator.state = (
                ck["rng_state"]
            )


            if "torch_rng" in ck:

                torch.set_rng_state(
                    ck["torch_rng"]
                )


            print(
                f"{name}: RESUMING "
                f"from epoch {start_epoch}"
            )


            print(
                f"best epoch="
                f"{best_epoch} | "
                f"best val="
                f"{best:.4e} | "
                f"wait={wait}"
            )


        except Exception as e:

            print(
                f"{name}: checkpoint load failed."
            )

            print(
                e
            )


    session_start = (
        time.perf_counter()
    )


    # -------------------------------------------------------------------------
    # Training
    # -------------------------------------------------------------------------

    for epoch in range(
        start_epoch,
        cfg.epochs+1
    ):

        net.train()


        for N,idx in schedule(
            TR,
            rng,
            True
        ):

            X = TR[N]["X"][
                idx
            ]


            Y = TR[N]["Y"][
                idx
            ]


            opt.zero_grad(
                set_to_none=True
            )


            L = batch_loss(
                net,
                X,
                Y
            )


            if not torch.isfinite(
                L
            ):

                raise RuntimeError(
                    f"{name}: non-finite loss"
                )


            L.backward()


            torch.nn.utils.clip_grad_norm_(

                net.parameters(),

                cfg.clip

            )


            opt.step()


        # ---------------------------------------------------------------------
        # Validation
        # ---------------------------------------------------------------------

        v = val_loss(
            net,
            VA
        )


        scheduler.step(
            v
        )


        # ---------------------------------------------------------------------
        # Early-stopping state
        # ---------------------------------------------------------------------

        if (
            best_state is None
            or
            v < best-cfg.min_delta
        ):

            best = v

            best_epoch = epoch

            wait = 0


            best_state = {

                k:
                x.detach().clone()

                for k,x
                in net.state_dict().items()

            }


        else:

            wait += 1


        # ---------------------------------------------------------------------
        # Permanent checkpoint
        # ---------------------------------------------------------------------

        if (
            epoch == 1
            or
            epoch % CHECKPOINT_EVERY == 0
        ):

            elapsed = (

                previous_elapsed

                +

                (
                    time.perf_counter()
                    -
                    session_start
                )

            )


            atomic_torch_save(

                {

                    "epoch":
                        epoch,

                    "model":
                        net.state_dict(),

                    "optimizer":
                        opt.state_dict(),

                    "scheduler":
                        scheduler.state_dict(),

                    "best":
                        best,

                    "best_epoch":
                        best_epoch,

                    "best_state":
                        best_state,

                    "wait":
                        wait,

                    "rng_state":
                        rng.bit_generator.state,

                    "torch_rng":
                        torch.get_rng_state(),

                    "elapsed_sec":
                        elapsed

                },

                ckpt_file

            )


            print(
                f"{name} | "
                f"epoch={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"best_epoch={best_epoch} | "
                f"wait={wait} | "
                "CHECKPOINT"
            )


        elif epoch % 20 == 0:

            print(
                f"{name} | "
                f"epoch={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"wait={wait}"
            )


        if wait >= cfg.patience:

            print(
                f"{name}: "
                f"early stopping at epoch "
                f"{epoch}"
            )

            break


    # -------------------------------------------------------------------------
    # Final model
    # -------------------------------------------------------------------------

    elapsed = (

        previous_elapsed

        +

        (
            time.perf_counter()
            -
            session_start
        )

    )


    if best_state is None:

        raise RuntimeError(
            f"{name}: no valid model state."
        )


    net.load_state_dict(
        best_state
    )


    atomic_torch_save(

        {

            "state":
                best_state,

            "best_epoch":
                best_epoch,

            "validation":
                best,

            "training_sec":
                elapsed

        },

        final_file

    )


    if ckpt_file.exists():

        ckpt_file.unlink()


    print(
        f"{name}: COMPLETE | "
        f"best epoch={best_epoch} | "
        f"best val={best:.4e} | "
        f"time={elapsed:.1f}s"
    )


    return (
        net,
        elapsed
    )


# =====================================================================================
# 26. TEST METRICS
# =====================================================================================

@torch.no_grad()
def evaluate(
    net,
    P
):

    net.eval()


    E2 = []

    Erho = []

    KL = []


    for N,G in P.items():

        for s in range(
            0,
            G["n"],
            cfg.batch
        ):

            X = G["X"][
                s:
                s+cfg.batch
            ]


            Y = G["Y"][
                s:
                s+cfg.batch
            ]


            Ph = predict_batch(
                net,
                X
            )


            # ---------------------------------------------------------
            # E2
            # ---------------------------------------------------------

            e2 = torch.linalg.vector_norm(
                Ph-Y,
                dim=1
            )


            # ---------------------------------------------------------
            # Tail error
            # ---------------------------------------------------------

            erho = torch.max(

                torch.abs(

                    tail_batch(Ph)

                    -

                    tail_batch(Y)

                ),

                dim=1

            ).values


            # ---------------------------------------------------------
            # KL
            # ---------------------------------------------------------

            Psafe = torch.clamp(
                Ph,
                min=1e-12,
                max=1.
            )


            pos = (
                Y>0
            )


            terms = torch.where(

                pos,

                Y
                *
                torch.log(
                    Y/Psafe
                ),

                torch.zeros_like(
                    Y
                )

            )


            kl = torch.sum(
                terms,
                dim=1
            )


            E2.append(
                e2.numpy()
            )


            Erho.append(
                erho.numpy()
            )


            KL.append(
                kl.numpy()
            )


    return (

        np.concatenate(
            E2
        ),

        np.concatenate(
            Erho
        ),

        np.concatenate(
            KL
        )

    )


# =====================================================================================
# 27. GENERATE DESIGNS
# =====================================================================================

print(
    "\nGenerating deterministic designs..."
)


train_design = design(
    cfg.R,
    cfg.trainN,
    cfg.seed+1
)


val_design = design(
    cfg.nval,
    cfg.trainN,
    cfg.seed+2
)


test_design = design(
    cfg.ntest,
    TEST_N,
    cfg.seed+3
)


# =====================================================================================
# 28. EXACT DATASETS — RESUMABLE
# =====================================================================================

print(
    "\nGenerating/loading exact targets..."
)


train = exact_set_resumable(
    train_design,
    "TRAIN"
)


val = exact_set_resumable(
    val_design,
    "VALIDATION"
)


test = exact_set_resumable(
    test_design,
    "TEST"
)


print(
    "\nPacking validation/test tensors..."
)


VA = pack(
    val
)


TE = pack(
    test
)


# =====================================================================================
# 29. PERMANENT EXPERIMENT PROGRESS
# =====================================================================================

PROGRESS_FILE = (
    ROOT
    /
    "experiment_progress.pkl"
)


progress = safe_pickle_load(
    PROGRESS_FILE,
    default=None
)


if progress is None:

    progress = {

        "exact":
            None,

        "mc":
            {}

    }


# =====================================================================================
# 30. IMPORT EXACT RESULT FROM PREVIOUS EXPERIMENT, IF AVAILABLE
# =====================================================================================

OLD_PROGRESS = (
    PREVIOUS_ROOT
    /
    "experiment_progress.pkl"
)


if (
    progress["exact"] is None
    and
    OLD_PROGRESS.exists()
):

    old_progress = safe_pickle_load(
        OLD_PROGRESS
    )


    if (
        old_progress is not None
        and
        old_progress.get(
            "exact"
        ) is not None
    ):

        progress["exact"] = (
            old_progress[
                "exact"
            ]
        )


        atomic_pickle(
            progress,
            PROGRESS_FILE
        )


        print(
            "\nExact-teacher metrics imported "
            "from previous compatible experiment."
        )


# =====================================================================================
# 31. EXACT-TEACHER NETWORK
# =====================================================================================

if progress["exact"] is None:

    print(
        "\n"
        +
        "="*90
    )

    print(
        "EXACT TEACHER | R=2000"
    )

    print(
        "="*90
    )


    TR_exact = pack(
        train
    )


    netE,exact_train_sec = fit_resumable(

        TR_exact,

        VA,

        "EXACT_R2000",

        cfg.seed+7001

    )


    E2_E,Erho_E,KL_E = evaluate(
        netE,
        TE
    )


    progress["exact"] = {

        "E2_med":
            float(
                np.median(
                    E2_E
                )
            ),

        "E2_q1":
            float(
                np.quantile(
                    E2_E,
                    .25
                )
            ),

        "E2_q3":
            float(
                np.quantile(
                    E2_E,
                    .75
                )
            ),

        "E2_p95":
            float(
                np.quantile(
                    E2_E,
                    .95
                )
            ),

        "Erho_med":
            float(
                np.median(
                    Erho_E
                )
            ),

        "Erho_q1":
            float(
                np.quantile(
                    Erho_E,
                    .25
                )
            ),

        "Erho_q3":
            float(
                np.quantile(
                    Erho_E,
                    .75
                )
            ),

        "Erho_p95":
            float(
                np.quantile(
                    Erho_E,
                    .95
                )
            ),

        "KL_med":
            float(
                np.median(
                    KL_E
                )
            )

    }


    atomic_pickle(
        progress,
        PROGRESS_FILE
    )


    print(
        "Exact-teacher result "
        "saved permanently."
    )


    del TR_exact
    del netE


else:

    print(
        "\nExact teacher already completed — skipping."
    )


EX = (
    progress[
        "exact"
    ]
)


# =====================================================================================
# 32. GENERATE / LOAD 50,000 MC TRAJECTORIES
# =====================================================================================

labels = mc_labels_resumable(
    train
)


# =====================================================================================
# 33. MONTE CARLO TEACHERS — RESUMABLE
# =====================================================================================

for m in M_VALUES:

    key = (
        f"m{m}"
    )


    # -------------------------------------------------------------------------
    # Result already completed
    # -------------------------------------------------------------------------

    if key in progress["mc"]:

        print(
            f"\nMC m={m}: "
            "already completed — skipping."
        )

        continue


    print(
        "\n"
        +
        "="*90
    )


    print(
        f"MONTE CARLO TEACHER | "
        f"R=2000 | m={m}"
    )


    print(
        "="*90
    )


    TR = pack(
        train,
        labels[m]
    )


    net,training_sec = fit_resumable(

        TR,

        VA,

        f"MC_R2000_m{m}",

        cfg.seed+7001

    )


    E2_M,Erho_M,KL_M = evaluate(
        net,
        TE
    )


    progress["mc"][key] = {

        "m":
            m,

        "E2_med":
            float(
                np.median(
                    E2_M
                )
            ),

        "E2_q1":
            float(
                np.quantile(
                    E2_M,
                    .25
                )
            ),

        "E2_q3":
            float(
                np.quantile(
                    E2_M,
                    .75
                )
            ),

        "E2_p95":
            float(
                np.quantile(
                    E2_M,
                    .95
                )
            ),

        "Erho_med":
            float(
                np.median(
                    Erho_M
                )
            ),

        "Erho_q1":
            float(
                np.quantile(
                    Erho_M,
                    .25
                )
            ),

        "Erho_q3":
            float(
                np.quantile(
                    Erho_M,
                    .75
                )
            ),

        "Erho_p95":
            float(
                np.quantile(
                    Erho_M,
                    .95
                )
            ),

        "KL_med":
            float(
                np.median(
                    KL_M
                )
            )

    }


    # -------------------------------------------------------------------------
    # Permanent completion checkpoint
    # -------------------------------------------------------------------------

    atomic_pickle(
        progress,
        PROGRESS_FILE
    )


    print(
        f"m={m}: result permanently saved."
    )


    print(
        f"E2 median="
        f"{np.median(E2_M):.6g} | "
        f"Erho median="
        f"{np.median(Erho_M):.6g} | "
        f"KL median="
        f"{np.median(KL_M):.6g}"
    )


    del TR
    del net


# =====================================================================================
# 34. FINAL SCIENTIFIC TABLE
#
# One table is sufficient for 5.3-A.
# =====================================================================================

table_rows = [

    {

        "Teacher":
            "Exact",

        "m":
            np.nan,

        **EX

    }

]


for m in M_VALUES:

    key = (
        f"m{m}"
    )


    if key in progress["mc"]:

        z = (
            progress[
                "mc"
            ][
                key
            ]
        )


        table_rows.append(

            {

                "Teacher":
                    "Monte Carlo",

                "m":
                    m,

                "E2_med":
                    z["E2_med"],

                "E2_q1":
                    z["E2_q1"],

                "E2_q3":
                    z["E2_q3"],

                "E2_p95":
                    z["E2_p95"],

                "Erho_med":
                    z["Erho_med"],

                "Erho_q1":
                    z["Erho_q1"],

                "Erho_q3":
                    z["Erho_q3"],

                "Erho_p95":
                    z["Erho_p95"],

                "KL_med":
                    z["KL_med"]

            }

        )


df = pd.DataFrame(
    table_rows
)


df.to_csv(

    OUT
    /
    "teacher_exact_vs_MC_R2000_MC50000.csv",

    index=False

)


(
    OUT
    /
    "teacher_exact_vs_MC_R2000_MC50000.tex"
).write_text(

    df.to_latex(

        index=False,

        float_format="%.4g"

    )

)


print(
    "\n"
    +
    "="*105
)


print(
    "FINAL 5.3-A TABLE"
)


print(
    "="*105
)


print(
    df.to_string(
        index=False
    )
)


# =====================================================================================
# 35. FINAL SCIENTIFIC FIGURE
#
# Panel A: median E2 + IQR
# Panel B: median E_rho + IQR
#
# Exact teacher = horizontal benchmark.
# =====================================================================================

mc = df[
    df["Teacher"]
    ==
    "Monte Carlo"
].copy()


if len(mc):

    fig,ax = plt.subplots(

        1,
        2,

        figsize=(
            10.5,
            4.2
        )

    )


    settings = [

        (
            ax[0],

            "E2_med",

            "E2_q1",

            "E2_q3",

            EX[
                "E2_med"
            ],

            r"$E_2$",

            "(A) Distributional error"
        ),

        (
            ax[1],

            "Erho_med",

            "Erho_q1",

            "Erho_q3",

            EX[
                "Erho_med"
            ],

            r"$E_\rho$",

            "(B) Tail-risk error"
        )

    ]


    for (
        a,
        y,
        q1,
        q3,
        exact_ref,
        ylabel,
        title

    ) in settings:


        x = mc[
            "m"
        ].to_numpy(
            dtype=float
        )


        med = mc[
            y
        ].to_numpy(
            dtype=float
        )


        lo = mc[
            q1
        ].to_numpy(
            dtype=float
        )


        hi = mc[
            q3
        ].to_numpy(
            dtype=float
        )


        # -------------------------------------------------------------
        # MC curve
        # -------------------------------------------------------------

        a.plot(

            x,

            med,

            "o-",

            color="#0072B2",

            lw=2,

            ms=6,

            label="Monte Carlo teacher"

        )


        # -------------------------------------------------------------
        # IQR
        # -------------------------------------------------------------

        a.fill_between(

            x,

            lo,

            hi,

            color="#0072B2",

            alpha=.15,

            linewidth=0

        )


        # -------------------------------------------------------------
        # Exact benchmark
        # -------------------------------------------------------------

        a.axhline(

            exact_ref,

            color="#D55E00",

            ls="--",

            lw=2,

            label="Exact teacher"

        )


        a.set_xscale(
            "log"
        )


        a.set_xticks(
            M_VALUES
        )


        a.set_xticklabels(
            M_VALUES
        )


        a.set_xlabel(
            r"Simulations per configuration $m$"
        )


        a.set_ylabel(
            ylabel
        )


        a.set_title(
            title
        )


        a.grid(
            alpha=.15
        )


        a.legend(
            frameon=False,
            fontsize=9
        )


    plt.tight_layout()


    plt.savefig(

        OUT
        /
        "teacher_exact_vs_MC_R2000_MC50000.pdf",

        bbox_inches="tight"

    )


    plt.savefig(

        OUT
        /
        "teacher_exact_vs_MC_R2000_MC50000.png",

        dpi=300,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================================
# 36. FINAL STATUS
# =====================================================================================

print(
    "\n"
    +
    "="*95
)


print(
    "EXPERIMENT 5.3-A STATUS"
)


print(
    "="*95
)


print(
    "Persistent Google Drive folder:"
)


print(
    ROOT
)


print()


print(
    "Exact teacher:",
    (
        "DONE"
        if progress["exact"] is not None
        else "NOT YET"
    )
)


for m in M_VALUES:

    print(
        f"MC m={m:2d}:",
        (
            "DONE"
            if f"m{m}" in progress["mc"]
            else "NOT YET"
        )
    )


print()


print(
    "R:",
    cfg.R
)


print(
    "m values:",
    M_VALUES
)


print(
    "Nominal nested MC trajectories:",
    f"{TOTAL_MC_EPISODES:,}"
)


print(
    "Maximum trajectories/configuration:",
    max(M_VALUES)
)


print(
    "Overflow early termination:",
    "ON"
)


print(
    "Exact chunk size:",
    EXACT_CHUNK
)


print(
    "MC chunk size:",
    MC_CHUNK
)


print(
    "Neural checkpoint interval:",
    f"{CHECKPOINT_EVERY} epochs"
)


print()


print(
    "Scientific table:"
)


print(
    OUT
    /
    "teacher_exact_vs_MC_R2000_MC50000.tex"
)


print(
    "Scientific figure:"
)


print(
    OUT
    /
    "teacher_exact_vs_MC_R2000_MC50000.pdf"
)


print(
    "\nAFTER A GOOGLE COLAB DISCONNECT:"
)


print(
    "1. Reconnect to Colab."
)


print(
    "2. Run THIS SAME CELL again."
)


print(
    "3. Existing exact chunks are loaded from Drive."
)


print(
    "4. Existing MC chunks are loaded from Drive."
)


print(
    "5. Completed models are skipped."
)


print(
    "6. Interrupted neural training resumes "
    "from the latest 5-epoch checkpoint."
)


print(
    "="*95
)

Mounted at /content/drive
PERMANENT GOOGLE DRIVE DIRECTORY
/content/drive/MyDrive/StatisticalLearning/Experiment_5_3A_R2000_MC50000_v4
CPU cores: 2
Exact sparse-LU workers: 2
Numba threads: 2
PyTorch threads: 2
Training configurations R: 2000
Monte Carlo m values: (1, 5, 10, 25)
Nested MC trajectories: 50,000

Generating deterministic designs...

Generating/loading exact targets...
TRAIN     0:   50 | computing with 2 workers
saved permanently | 74.8s
TRAIN    50:  100 | computing with 2 workers
saved permanently | 62.7s
TRAIN   100:  150 | computing with 2 workers
saved permanently | 80.0s
TRAIN   150:  200 | computing with 2 workers
saved permanently | 61.0s
TRAIN   200:  250 | computing with 2 workers
saved permanently | 64.9s
TRAIN   250:  300 | computing with 2 workers
saved permanently | 63.0s
TRAIN   300:  350 | computing with 2 workers
saved permanently | 34.7s
TRAIN   350:  400 | computing with 2 workers
saved permanently | 60.0s
TRAIN   400:  450 | computing with 2 workers
